# Optimization — MIPROv2

MIPROv2 is DSPy's full optimizer. Unlike `BootstrapFewShot` which only selects few-shot examples, MIPROv2:
1. Proposes many candidate **instruction strings** for each signature
2. Selects the best **few-shot demonstrations** from your trainset
3. Searches over combinations using a Bayesian optimizer

We use **GSM8K** (grade school math) because instruction wording measurably changes how the model reasons through multi-step problems — making the optimization delta easy to see.

In [1]:
import re
import random
import dspy
from datasets import load_dataset
from dotenv import load_dotenv
load_dotenv()

lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

## Step 1 — Load GSM8K and extract final answers

Each answer in GSM8K has a chain-of-thought followed by `#### <number>`.  
We only keep the final number as the ground truth label.

In [3]:
raw_train = load_dataset('gsm8k', 'main', split='train')
raw_test  = load_dataset('gsm8k', 'main', split='test')

def extract_answer(raw_answer: str) -> str:
    """Pull the numeric answer after #### ."""
    return raw_answer.split('#### ')[-1].strip()

def make_examples(split):
    return [
        dspy.Example(
            question=row['question'],
            answer=extract_answer(row['answer'])
        ).with_inputs('question')
        for row in split
    ]

all_train = make_examples(raw_train)
all_test  = make_examples(raw_test)

random.seed(42)
random.shuffle(all_train)

trainset = all_train[:200]
devset   = all_test[:100]

print(f'Train: {len(trainset)} | Dev: {len(devset)}')
print(f'Example Q: {devset[0].question}')
print(f'Example A: {devset[0].answer}')

Train: 200 | Dev: 100
Example Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Example A: 18


## Step 2 — Define the module

`ChainOfThought` tells DSPy to ask the model to reason before answering.  
The signature asks for a final numeric `answer` — no units, no explanation.

In [5]:
class MathSolver(dspy.Signature):
    """Solve the math word problem. Return only the final numeric answer."""

    question: str = dspy.InputField()
    answer: str   = dspy.OutputField(desc="Final numeric answer only, no units or explanation")


class Solver(dspy.Module):
    def __init__(self):
        self.solve = dspy.ChainOfThought(MathSolver)

    def forward(self, question):
        return self.solve(question=question)

## Step 3 — Write the metric

We strip commas and whitespace then compare strings.  
GSM8K answers are always integers so exact string match is sufficient.

In [6]:
def parse_number(text: str) -> str:
    """Extract trailing integer or decimal, strip commas."""
    text = text.replace(',', '').strip()
    numbers = re.findall(r'-?\d+\.?\d*', text)
    return numbers[-1] if numbers else text

def math_accuracy(example, prediction, trace=None):
    expected = parse_number(example.answer)
    predicted = parse_number(prediction.answer)
    return float(expected == predicted)

## Step 4 — Baseline evaluation

In [14]:
from dspy.evaluate import Evaluate

evaluate = Evaluate(devset=devset, metric=math_accuracy, num_threads=4, display_progress=True)

baseline = Solver()
baseline_score = evaluate(baseline)
print(f'Baseline accuracy: {baseline_score.score:.1f}%')

Average Metric: 92.00 / 100 (92.0%): 100%|██████████| 100/100 [00:00<00:00, 204.94it/s]

2026/09/03 14:23:30 INFO dspy.evaluate.evaluate: Average Metric: 92.0 / 100 (92.0%)



Baseline accuracy: 92.0%


## Step 5 — Optimize with MIPROv2

`auto="medium"` runs a broader search — more candidate instructions, more Bayesian trials.  
Expect ~15 minutes with gpt-4o-mini and ~$0.50 in API cost.  
MIPROv2 will propose new instruction strings and search for the best combination.

In [16]:
import logging
import os
from contextlib import redirect_stderr
from dspy.teleprompt import MIPROv2

logging.getLogger("dspy").setLevel(logging.WARNING)

optimizer = MIPROv2(
    metric=math_accuracy,
    auto='medium',
    verbose=False,
)

with open(os.devnull, 'w') as devnull, redirect_stderr(devnull):
    optimized = optimizer.compile(
        Solver(),
        trainset=trainset,
        requires_permission_to_run=False,
    )

print('Optimization complete.')

Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12
Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/12
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 5/12
Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Bootstrapping set 6/12
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 7/12
Bootstrapped 2 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Bootstrapping set 8/12
Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 9/12
Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Bootstrapping set 10/12
Bootstrapped 1 full traces after 1 examples for up to 1 rounds, amounting to 1 attempts.
Bootstrapping set 11/12
Bootstrapped 2 full traces after 

## Step 6 — Compare results

In [17]:
optimized_score = evaluate(optimized)

print(f'Baseline : {baseline_score.score:.1f}%')
print(f'Optimized: {optimized_score.score:.1f}%')
print(f'Delta    : +{optimized_score.score - baseline_score.score:.1f}%')

Average Metric: 94.00 / 100 (94.0%): 100%|██████████| 100/100 [00:00<00:00, 324.78it/s]
Baseline : 92.0%
Optimized: 94.0%
Delta    : +2.0%


## Step 7 — Inspect what MIPROv2 wrote

See the instruction text the optimizer generated — this is the prompt engineering it did automatically.

In [13]:
optimized.solve.predict.signature

StringSignature(question -> reasoning, answer
    instructions='When presented with a math word problem, analyze the scenario and follow a structured reasoning process to break down the calculations step by step. Compute the necessary values and derive the final numeric answer without providing supplementary explanations or units.'
    question = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Question:', 'desc': '${question}'})
    reasoning = Field(annotation=str required=True json_schema_extra={'desc': '${reasoning}', '__dspy_field_type': 'output', 'prefix': 'Reasoning:'})
    answer = Field(annotation=str required=True json_schema_extra={'desc': 'Final numeric answer only, no units or explanation', '__dspy_field_type': 'output', 'prefix': 'Answer:'})
)

In [12]:
optimized(question="A train travels 60 miles per hour. How far does it travel in 2.5 hours?")
dspy.inspect_history(n=1)





[2026-08-25T10:25:36.251854]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `reasoning` (str): 
2. `answer` (str): Final numeric answer only, no units or explanation
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        When presented with a math word problem, analyze the scenario and follow a structured reasoning process to break down the calculations step by step. Compute the necessary values and derive the final numeric answer without providing supplementary explanations or units.


User message:

This is an example of the task, though some input or output fields are not supplied.

[[ ## question ## ]]
Jason's dog has a tail that's half the length of its body, and a head that's 1/6 the length of its body. If the dog is 

## Prompt Diff — Before vs After MIPROv2

**Before (hand-written):**
```
Solve the math word problem. Return only the final numeric answer.

Question: {question}
Reasoning: {reasoning}
Answer: {answer}
```

**After (MIPROv2-generated):**
```
When presented with a math word problem, analyze the scenario and follow a structured
reasoning process to break down the calculations step by step. Compute the necessary
values and derive the final numeric answer without providing supplementary explanations
or units.

Question: {question}
Reasoning: {reasoning}
Answer: {answer}
```

**Result:** 92.0% → 94.0% (+2.0%) on GSM8K dev set (100 examples, gpt-4o-mini, `auto="medium"`)